### Key Takeaways

**Exact Match (EM)**:
- Very strict metric - requires perfect answer
- Good for measuring when you need precise answers
- Can be 0 even if answer is mostly correct

**F1 Score**:
- Measures token overlap - gives partial credit
- Better for understanding how close answers are
- Range: 0 (no overlap) to 1 (perfect match)

**Confidence Scores**:
- Sum of start and end logit scores
- Higher confidence usually (but not always) means better answer
- Useful for filtering low-confidence predictions

**Why These Metrics Matter**:
- They help us quantify model performance
- Essential for comparing different models or approaches
- Guide decisions on when a model is "good enough" for production

In a production Q&A system, you typically:
1. Set a minimum confidence threshold (e.g., reject answers with confidence < X)
2. Track EM and F1 on a validation set over time
3. Use these metrics to decide when to retrain or switch models

---

This concludes our exploration of Question Answering with BERT and its evaluation metrics!

In [ ]:
# Test dataset: questions, contexts, and ground truth answers
qa_test_data = [
    {
        'question': 'What is the immune system?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'a system of many biological structures and processes within an organism that protects against disease'
    },
    {
        'question': 'What must the immune system detect?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'a wide variety of agents, known as pathogens'
    },
    {
        'question': 'What are pathogens?',
        'context': 'The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism\'s own healthy tissue.',
        'answer': 'agents from viruses to parasitic worms'
    }
]

def get_answer(question, context):
    """
    Extract answer from context using BERT Q&A model.
    """
    # Prepare input
    question_text = '[CLS] ' + question + '[SEP]'
    context_text = context + '[SEP]'
    
    # Tokenize
    question_tokens = tokenizer.tokenize(question_text)
    context_tokens = tokenizer.tokenize(context_text)
    
    # Combine and convert to IDs
    tokens = question_tokens + context_tokens
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    
    # Segment IDs
    segment_ids = [0] * len(question_tokens) + [1] * len(context_tokens)
    
    # Convert to tensors
    input_ids = torch.tensor([input_ids])
    segment_ids = torch.tensor([segment_ids])
    
    # Get model output
    output = model(input_ids=input_ids, token_type_ids=segment_ids)
    
    # Get start and end indices
    start_index = torch.argmax(output.start_logits)
    end_index = torch.argmax(output.end_logits)
    
    # Extract answer
    answer = ' '.join(tokens[start_index:end_index+1])
    
    # Get confidence scores
    start_score = torch.max(output.start_logits).item()
    end_score = torch.max(output.end_logits).item()
    confidence = start_score + end_score
    
    return answer, confidence

# Evaluate on test data
print("Evaluating Q&A Model:\n")
print("="*80)

results = []

for i, item in enumerate(qa_test_data, 1):
    predicted_answer, confidence = get_answer(item['question'], item['context'])
    
    em = calculate_exact_match(predicted_answer, item['answer'])
    f1 = calculate_f1(predicted_answer, item['answer'])
    
    results.append({
        'em': em,
        'f1': f1,
        'confidence': confidence
    })
    
    print(f"\nExample {i}:")
    print(f"Question: {item['question']}")
    print(f"Predicted: {predicted_answer}")
    print(f"Truth: {item['answer']}")
    print(f"EM: {em}  |  F1: {f1:.3f}  |  Confidence: {confidence:.2f}")

print("\n" + "="*80)

# Calculate average metrics
avg_em = sum(r['em'] for r in results) / len(results)
avg_f1 = sum(r['f1'] for r in results) / len(results)
avg_confidence = sum(r['confidence'] for r in results) / len(results)

print(f"\n📊 Average Performance:")
print(f"  Exact Match: {avg_em:.2%}")
print(f"  F1 Score: {avg_f1:.3f}")
print(f"  Avg Confidence: {avg_confidence:.2f}")

print("\n" + "="*80)

### Evaluate on Multiple Examples

Now let's test our Q&A model on several question-context pairs and calculate average metrics.

In [ ]:
def normalize_answer(text):
    """
    Normalize answer text for comparison by:
    - Converting to lowercase
    - Removing punctuation
    - Removing articles (a, an, the)
    - Removing extra whitespace
    """
    import re
    import string
    
    # Lowercase
    text = text.lower()
    
    # Remove punctuation
    text = ''.join(ch if ch not in string.punctuation else ' ' for ch in text)
    
    # Remove articles
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

def calculate_exact_match(prediction, ground_truth):
    """
    Calculate Exact Match score.
    Returns 1 if normalized strings match exactly, 0 otherwise.
    """
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def calculate_f1(prediction, ground_truth):
    """
    Calculate F1 score based on token overlap.
    """
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    
    common_tokens = set(pred_tokens) & set(truth_tokens)
    
    if len(common_tokens) == 0:
        return 0
    
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(truth_tokens)
    
    f1 = 2 * (precision * recall) / (precision + recall)
    
    return f1

# Test the metrics with examples
print("Testing Metrics:\n")
print("="*80)

test_cases = [
    ("a system of many biological structures", "a system of many biological structures", "Perfect match"),
    ("system of biological structures", "a system of many biological structures", "Missing words"),
    ("the biological system", "a system of many biological structures", "Partial match"),
    ("completely wrong answer", "a system of many biological structures", "No match"),
]

for pred, truth, description in test_cases:
    em = calculate_exact_match(pred, truth)
    f1 = calculate_f1(pred, truth)
    print(f"\n{description}:")
    print(f"  Predicted: '{pred}'")
    print(f"  Truth: '{truth}'")
    print(f"  EM: {em}  |  F1: {f1:.3f}")

print("\n" + "="*80)

# Q&A with finetuned BERT

In this section, let's learn how to perform question answering with a finetuned Q&A BERT. First, let us import the necessary modules:

In [23]:
%%capture
!pip install transformers==3.5.1

In [24]:
import torch
from transformers import BertForQuestionAnswering, BertTokenizer


Now, we download and load the model. We use the bert-large-uncased-whole-word-masking-finetuned-squad model which is finetuned on the SQUAD (Stanford question answering dataset).


In [25]:
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



Next, we download and load the tokenizer which is used for pretraining the bert-large-uncased-whole-word-masking-finetuned-squad model:


In [26]:
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')


Now that we downloaded the model and tokenizer, let's preprocess the input.

## Preprocessing the input
First, we define the input to the BERT which is question and paragraph text:


In [27]:
question = "What is the immune system?"
paragraph = "The immune system is a system of many biological structures and processes within an organism that protects against disease. To function properly, an immune system must detect a wide variety of agents, known as pathogens, from viruses to parasitic worms, and distinguish them from the organism's own healthy tissue."

Add [CLS] token to the beginning of the question and [SEP] token at the end of both the question and paragraph:

In [28]:
question = '[CLS] ' + question + '[SEP]'
paragraph = paragraph + '[SEP]'


Now, tokenize the question and paragraph:


In [29]:
question_tokens = tokenizer.tokenize(question)
paragraph_tokens = tokenizer.tokenize(paragraph)



Combine the question and paragraph tokens and convert them to input_ids:

In [30]:
tokens = question_tokens + paragraph_tokens
input_ids = tokenizer.convert_tokens_to_ids(tokens)



Next, we define the segment_ids. The segment_ids will be 0 for all the tokens of question and it will be 1 for all the tokens of the paragraph:


In [31]:
segment_ids = [0] * len(question_tokens)
segment_ids += [1] * len(paragraph_tokens)


Now we convert the input_ids and segment_ids to tensor:

In [32]:
input_ids = torch.tensor([input_ids])
segment_ids = torch.tensor([segment_ids])



Now that we processed the input. Let's feed them to the model and get the result.

## Getting the answer
We feed the input_ids and segment_ids to the model which return the start score and end score for all of the tokens:


In [35]:
output = model(input_ids=input_ids, token_type_ids = segment_ids)

In [37]:
start_scores = output.start_logits
end_scores = output.end_logits


Now, we select the start_index which is the index of the token which has a maximum start score and end_index which is the index of the token which has a maximum end score:


In [38]:
start_index = torch.argmax(start_scores)
end_index = torch.argmax(end_scores)


That's it! Now, we print the text span between the start and end index as our answer:

In [39]:
print(' '.join(tokens[start_index:end_index+1]))

a system of many biological structures and processes within an organism that protects against disease


---

## Evaluating Q&A Performance with Metrics

The example above shows how to get a single answer, but how do we measure if our model is performing well? Let's look at key metrics for Question Answering systems.

### Understanding Q&A Metrics

For extractive Question Answering (like BERT Q&A), we use two main metrics:

1. **Exact Match (EM)**: The predicted answer must match the ground truth exactly (after normalization)
   - Binary: either 1 (perfect match) or 0 (no match)
   - Strict but clear measure of success

2. **F1 Score**: Token-level overlap between prediction and ground truth
   - Measures partial credit for partially correct answers
   - More lenient than EM

Let's implement these metrics and test them on multiple examples.